In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
import tifffile as tiff
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

from skimage import io
import sys
# from umap import UMAP
import joblib
from datetime import datetime

# Load Data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [11]:
# Get the directory of the script
script_dir = os.getcwd()

# Get the parent directory of the script
parent_dir = os.path.dirname(script_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)

from core.autoencoders import AE, train_ae, VAE32, train_vae
from core.dataset import TIFFDataset
from utils.feature_analysis import dataloader_AE_VAE_latents,UMAP_train
from utils.plotting_utils import umap_2Dplot


ImportError: cannot import name 'dataloader_AE_VAE_latents' from 'utils.feature_analysis' (/mnt/d/lding/CLS_GitHub/fa_patch_AE_clustering/utils/feature_analysis.py)

In [3]:
data_str = 'vin_pax_zyx_act_wholecell'
pro_str = 'pax_wholecell_norm_correctloader'
main_ch = 1
ctrl_y_str = 'ctrl_y'
input_ps = 32
latent_dim = 8
BN_flag = True
dropout_flag = True
epochs = int(1000)
lr = 1e-4
loss_norm_flag = 1

dir_list = [
'/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/ctrl_ch1_major/ctrl_ch1_patches_gridonly_wholecell_pslocation00/tiff_patches32_40p_20250919_0948',
'/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/y_ch1_major/y_ch1_patches_gridonly_wholecell_pslocation00/tiff_patches32_40p_20250919_1005',    
]


In [4]:
# Define preprocessing transforms here.
# Normalization is usually included as part of the transform pipeline,
# which is passed into the Dataset during initialization.

# this setting is with no normalization
v_min = 0
v_max = 1

transform = transforms.Compose([
    transforms.ToTensor(),
    lambda x: (x-v_min)/(v_max-v_min)
])

# load and combine datasets
datasets = [
    TIFFDataset(root_dir=dir_path, label=label, transform=transform)
    for label, dir_path in enumerate(dir_list)
]

combined_dataset = ConcatDataset(datasets)

# Split dataset into training and validation sets
train_size = int(0.8 * len(combined_dataset))
val_size = len(combined_dataset) - train_size
train_dataset, val_dataset = random_split(combined_dataset, [train_size, val_size])
whole_data_loader = DataLoader(combined_dataset, batch_size=128, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)



/mnt/d/lding/CLS_GitHub/fa_patch_AE_clustering/core/dataset.py:31: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  image = torch.tensor(image, dtype=torch.float32)  # shape: (1, H, W)


In [5]:
now = datetime.now()

time_str = now.strftime("%Y%m%d_%H%M")

result_dir = os.path.join('../results/', pro_str+'_ch' + str(main_ch) +'_'+ 'ps'+str(input_ps)+'_latdim_'+ str(latent_dim) +'_'+ time_str)
os.makedirs(result_dir, exist_ok=True)

# Define AE
no_ch = 1
vae = VAE32(in_channels=no_ch, latent_dim=latent_dim).to(device)
vae, tr, va = train_vae(
    vae, train_loader, val_loader, device,
    epochs=epochs, lr=lr,
    beta=1.0, recon_type="mse",
    result_dir=result_dir
)


Epoch 20/1000 | Train: total=0.0323 recon=0.0320 kl=0.0003 | Val: total=0.0329 recon=0.0326 kl=0.0003
Epoch 40/1000 | Train: total=0.0321 recon=0.0319 kl=0.0001 | Val: total=0.0327 recon=0.0326 kl=0.0002
Epoch 60/1000 | Train: total=0.0320 recon=0.0319 kl=0.0001 | Val: total=0.0327 recon=0.0326 kl=0.0001
Epoch 80/1000 | Train: total=0.0320 recon=0.0319 kl=0.0000 | Val: total=0.0326 recon=0.0326 kl=0.0000
Epoch 100/1000 | Train: total=0.0320 recon=0.0319 kl=0.0000 | Val: total=0.0326 recon=0.0326 kl=0.0001
Epoch 120/1000 | Train: total=0.0320 recon=0.0319 kl=0.0000 | Val: total=0.0326 recon=0.0325 kl=0.0000
Epoch 140/1000 | Train: total=0.0320 recon=0.0319 kl=0.0000 | Val: total=0.0326 recon=0.0325 kl=0.0000
Epoch 160/1000 | Train: total=0.0320 recon=0.0319 kl=0.0000 | Val: total=0.0326 recon=0.0325 kl=0.0000
Epoch 180/1000 | Train: total=0.0320 recon=0.0319 kl=0.0000 | Val: total=0.0325 recon=0.0325 kl=0.0000
Epoch 200/1000 | Train: total=0.0320 recon=0.0319 kl=0.0000 | Val: total=0.03

In [ ]:
# obtain the latent features
latents, images, group_id = dataloader_AE_VAE_latents(vae, whole_data_loader, device, 'mu')

if isinstance(group_id, torch.Tensor):
    group_id = group_id.cpu().numpy()
elif isinstance(group_id, list):
    group_id = torch.cat(group_id).cpu().numpy()
    
print(latents.shape)
print(images.shape)


TypeError: dataloader_model_latents() takes 3 positional arguments but 4 were given

In [ ]:
umap_model_name = "vae_vin_pax_zyx_act_wholecell_umap"

latents_2d = UMAP_train(latents, result_dir,umap_model_name)

fig = umap_2Dplot(latents_2d, 0,1,group_id)
    